In [ ]:
# Cell 1: Setup

# Import necessary libraries
import pandas as pd
import torch
import numpy as np
from transformers import BertTokenizer, BertModel
from sklearn.preprocessing import MultiLabelBinarizer
import re

# Check if GPU is available and set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Function to preprocess log entries
def preprocess_log_entry(log_entry):
    """
    Parses a raw log entry to extract and normalize key entities.
    """
    # Pattern for the new log format
    log_pattern = re.compile(
        r'^-?\s*'  # Optional leading dash and whitespace
        r'(\d+)\s+'  # Group 1: A number (like an ID)
        r'(\d{4}\.\d{2}\.\d{2})\s+'  # Group 2: Date
        r'([\w\-:]+)\s+'  # Group 3: Source component
        r'(\d{4}-\d{2}-\d{2}-\d{2}\.\d{2}\.\d{2}\.\d+)\s+'  # Group 4: Timestamp
        r'([\w\-:]+)\s+'  # Group 5: Repeated source component
        r'(.*)'  # Group 6: The rest of the message
    )

    match = log_pattern.match(log_entry)

    if match:
        return {
            "timestamp": match.group(4),
            "source_component": match.group(3),
            "message": match.group(6).strip(),
        }
    else:
        return {
            "timestamp": "N/A",
            "source_component": "N/A",
            "message": log_entry,  # Return the original entry if no match
        }


# Function to get BERT embeddings
def get_bert_embeddings(token_ids, model):
    """
    Generates BERT embeddings for a given set of token IDs.

    Args:
        token_ids (torch.Tensor): A tensor of token IDs.
        model: A pre-trained BERT model.

    Returns:
        np.ndarray: The embedding vector for the [CLS] token.
    """
    with torch.no_grad():
        outputs = model(input_ids=token_ids.unsqueeze(0).to(device))
        cls_embedding = outputs.last_hidden_state[0, 0, :].cpu().numpy()
    return cls_embedding

# Function to map actions to UCO concepts
def map_to_uco(actions):
    """
    Maps extracted log actions to simplified UCO concepts using a rule-based approach.

    Args:
        actions (list): A list of action strings extracted from a log.

    Returns:
        list: A list of corresponding UCO class labels.
    """

    uco_map = {
        # Authentication Actions
        'login': 'uco-action:LogonAction',
        'logon': 'uco-action:LogonAction',
        'failed': 'uco-action:AuthenticationAction-Failed',
        'invalid': 'uco-action:AuthenticationAction-Failed',
        'success': 'uco-action:AuthenticationAction-Success',
        'accepted': 'uco-action:AuthenticationAction-Success',
        'logout': 'uco-action:LogoffAction',
        'timeout': 'uco-action:AuthenticationAction-Timeout',
        'password': 'uco-action:AuthenticationAction-PasswordChange',
        'login attempt': 'uco-action:AuthenticationAction-Login',
        'bruteforce': 'uco-action:AuthenticationAction-Bruteforce',
        'unauthorized': 'uco-action:AccessAction-Denied',
        'privilege escalation': 'uco-action:PrivilegeEscalation',

        # Process Actions
        'starting': 'uco-action:ProcessAction-Start',
        'started': 'uco-action:ProcessAction-Start',
        'launch': 'uco-action:ProcessAction-Start',
        'run': 'uco-action:ProcessAction-Start',
        'executed': 'uco-action:ProcessAction-Start',
        'terminating': 'uco-action:ProcessAction-Terminate',
        'terminated': 'uco-action:ProcessAction-Terminate',
        'stopped': 'uco-action:ProcessAction-Terminate',
        'killed': 'uco-action:ProcessAction-Terminate',
        'process creation': 'uco-action:ProcessAction-Start',
        'process termination': 'uco-action:ProcessAction-Terminate',

        # File Actions
        'read': 'uco-action:FileAction-Read',
        'open': 'uco-action:FileAction-Read',
        'write': 'uco-action:FileAction-Write',
        'modify': 'uco-action:FileAction-Modify',
        'update': 'uco-action:FileAction-Modify',
        'delete': 'uco-action:FileAction-Delete',
        'removed': 'uco-action:FileAction-Delete',
        'created': 'uco-action:FileAction-Create',
        'copied': 'uco-action:FileAction-Copy',
        'moved': 'uco-action:FileAction-Move',
        'renamed': 'uco-action:FileAction-Rename',
        'file access': 'uco-action:FileAction-Read',
        'file modification': 'uco-action:FileAction-Modify',

        # Network Actions
        'connect': 'uco-action:NetworkAction-Connect',
        'connection': 'uco-action:NetworkAction-Connect',
        'disconnect': 'uco-action:NetworkAction-Disconnect',
        'upload': 'uco-action:NetworkAction-Upload',
        'download': 'uco-action:NetworkAction-Download',
        'sent': 'uco-action:NetworkAction-Send',
        'received': 'uco-action:NetworkAction-Receive',
        'network connection': 'uco-action:NetworkAction-Connect',
        'ip blocked': 'uco-action:NetworkAction-Block',
        'ip allowed': 'uco-action:NetworkAction-Allow',
        'ip denied': 'uco-action:NetworkAction-Deny',

        # DNS Actions
        'dns query': 'uco-action:DNSAction-Query',
        'dns response': 'uco-action:DNSAction-Response',

        # HTTP Actions
        'http get': 'uco-action:HTTPAction-Get',
        'http post': 'uco-action:HTTPAction-Post',
        'http put': 'uco-action:HTTPAction-Put',
        'http delete': 'uco-action:HTTPAction-Delete',

        # Observables
        'ip address': 'uco-observable:IPv4Address',
        'ipv4': 'uco-observable:IPv4Address',
        'ipv6': 'uco-observable:IPv6Address',
        'src ip': 'uco-observable:IPv4Address',
        'dst ip': 'uco-observable:IPv4Address',
        'source ip': 'uco-observable:IPv4Address',
        'destination ip': 'uco-observable:IPv4Address',
        'port': 'uco-observable:Port',
        'protocol': 'uco-observable:Protocol',
        'network interface': 'uco-observable:NetworkInterface',
        'file': 'uco-observable:File',
        'file path': 'uco-observable:FilePath',
        'process': 'uco-observable:Process',
        'process id': 'uco-observable:ProcessID',
        'authentication token': 'uco-observable:AuthenticationToken',
        'dns record': 'uco-observable:DNSRecord',
        'dns name': 'uco-observable:DNSName',
        'http request': 'uco-observable:HTTPRequest',
        'http response': 'uco-observable:HTTPResponse',

        # States
        'locked': 'uco-state:AccountLocked',
        'error': 'uco-state:Error',
        'warning': 'uco-state:Warning',
        'critical': 'uco-state:Critical',
        'info': 'uco-state:Info',
        'alert': 'uco-state:Alert',
        'denied': 'uco-action:AccessAction-Denied',
    }

    uco_labels = [uco_map[action] for action in actions if action in uco_map]
    return list(set(uco_labels))  # Return unique labels

# Print a message indicating that the setup is complete
print("Setup complete. Ready to process logs.")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 3: Load Raw Data

# Function to load raw log data
def load_raw_logs(file_path):
    """
    Loads raw log data from a specified file path.

    Args:
        file_path (str): The path to the log file.

    Returns:
        pd.DataFrame: A DataFrame containing the loaded log data.
    """
    print(f"--- Loading raw log data from {file_path} ---")
    try:
        # Assuming the file is a text file based on the extension and error
        # Reading as a single column DataFrame
        log_df = pd.read_csv(file_path, header=None, names=['log'], on_bad_lines='skip')

        print("Raw log data loaded successfully.")
        return log_df

    except FileNotFoundError:
        print(f"Error: File not found at {file_path}")
        return pd.DataFrame() # Return an empty DataFrame in case of error
    except Exception as e:
        print(f"Error loading log file: {e}")
        return pd.DataFrame() # Return an empty DataFrame in case of error


# Example usage: Replace with your actual log file path
file_path =  "/content/drive/MyDrive/BGL/BGL.log"
global log_df # Make log_df a global variable
log_df = load_raw_logs(file_path)

# Display the first few rows of the DataFrame
print("\n--- Loaded Raw Log Data ---")
if not log_df.empty:
    print(log_df.head())
    print(f"\nTotal number of logs loaded: {len(log_df)}")
else:
    print("No log data loaded.")

In [ ]:
# Cell 4: Preprocessing (Step 1)

# Function to preprocess the entire DataFrame
def preprocess_logs(log_df):
    """
    Applies the preprocessing function to the entire DataFrame.

    Args:
        log_df (pd.DataFrame): DataFrame containing raw log data.

    Returns:
        pd.DataFrame: DataFrame with extracted entities.
    """
    if log_df.empty:
        print("No data to preprocess.")
        return pd.DataFrame()

    print("\n--- Preprocessing Log Entries ---")
    # Apply the preprocessing function to each log entry in the 'log' column
    processed_data = log_df['log'].apply(preprocess_log_entry)

    # Convert the list of dictionaries into a DataFrame
    processed_df = pd.json_normalize(processed_data)

    # Join with the original DataFrame to keep the 'log' column
    processed_df = log_df.join(processed_df)


    return processed_df

# Apply the preprocessing function to the loaded log DataFrame
processed_log_df = preprocess_logs(log_df)

# Display the first few rows of the processed DataFrame
print("\n--- Processed Log Data ---")
if not processed_log_df.empty:
    print(processed_log_df.head())
    print(f"\nTotal number of processed logs: {len(processed_log_df)}")
else:
    print("No processed log data.")

In [ ]:
# Cell 5: Semantic Enrichment (Step 2) & Sentence Construction (Step 4)

# Function to perform semantic enrichment and sentence construction
def semantic_enrichment_and_sentence_construction(log_df, batch_size=32):
    """
    Enriches the log DataFrame with extracted actions, UCO labels, and constructs
    semantic sentences.

    Args:
        log_df (pd.DataFrame): DataFrame containing preprocessed logs.
        batch_size (int): Batch size for potential future steps (not directly used here).

    Returns:
        pd.DataFrame: Enriched DataFrame with extracted actions, UCO labels, and semantic sentences.
    """
    if log_df.empty:
        print("No data to enrich and construct sentences from.")
        return pd.DataFrame()

    # Extract actions and map to UCO concepts
    print("\n--- Extracting actions and mapping to UCO concepts ---")
    # Ensure the 'message' column exists and is not empty
    if 'message' not in log_df.columns or log_df['message'].isnull().all():
        print("'message' column not found or is empty. Cannot extract actions.")
        log_df['actions'] = [[]] * len(log_df)
        log_df['uco_labels'] = [[]] * len(log_df)
    else:
        log_df['actions'] = log_df['message'].apply(lambda x: re.findall(r'\b(login|logon|failed|invalid|success|accepted|denied|logout|timeout|password|starting|started|launch|run|executed|terminating|terminated|stopped|killed|read|open|write|modify|update|delete|removed|created|copied|moved|renamed|connect|connection|disconnect|upload|download|sent|received|unauthorized|privilege escalation|bruteforce|locked|error|warning|critical|info|alert)\b', str(x).lower()))
        log_df['uco_labels'] = log_df['actions'].apply(map_to_uco)

    # Construct semantic sentences
    print("\n--- Constructing Semantic Sentences ---")
    def construct_semantic_sentence(row):
        """
        Constructs a semantically meaningful sentence from a processed log entry row.
        Adapt this based on the available columns and desired sentence structure.
        """
        timestamp = row.get('timestamp', 'N/A')
        source = row.get('source_component', 'N/A')
        message = row.get('message', '')
        actions = ", ".join(row.get('actions', [])) or "an event"

        # Example sentence structure - modify as needed
        sentence = f"At {timestamp}, the component {source} recorded {actions} related to: {message}"
        return sentence

    # Apply sentence construction
    log_df['semantic_sentence'] = log_df.apply(construct_semantic_sentence, axis=1)

    return log_df

# Apply the semantic enrichment and sentence construction function
enriched_log_df = semantic_enrichment_and_sentence_construction(processed_log_df)

# Display the first few rows of the enriched DataFrame with new columns
print("\n--- Enriched Log Data with Actions, UCO Labels, and Semantic Sentences ---")
if not enriched_log_df.empty:
    # Select columns to display, ensuring they exist
    display_cols = ['log', 'timestamp', 'source_component', 'message', 'actions', 'uco_labels', 'semantic_sentence']
    existing_cols = [col for col in display_cols if col in enriched_log_df.columns]
    print(enriched_log_df[existing_cols].head())
    print(f"\nTotal number of enriched logs: {len(enriched_log_df)}")
else:
    print("No enriched log data.")

In [ ]:
# Cell 6: Embedding Generation with Cybersecurity-aware BERT (Step 5)

# This cell assumes you have access to a pre-trained Cybersecurity-aware BERT model.
# You might need to install a specific library or download the model weights.
# Replace 'cybersecurity-bert-model' with the actual model name or path.

# Example using a hypothetical model name - replace with the actual model
# A more appropriate model for cybersecurity tasks might be 'bert-base-uncased' fine-tuned on cybersecurity data,
# or a model specifically trained for this domain if available.
CYBER_BERT_MODEL ='jackaduma/SecBERT' # Replace with the actual model name if different

from tqdm.notebook import tqdm # Import tqdm for progress bar
import time # Import time for speed calculation
import os # Import os for file operations
import re # Import re for regular expressions
from sklearn.preprocessing import MultiLabelBinarizer # Import MultiLabelBinarizer
import pandas as pd # Import pandas
import numpy as np # Import numpy
import torch # Import torch

def generate_cyber_bert_embeddings_and_hybrid_vectors(log_df, model_name=CYBER_BERT_MODEL, batch_size=32, segment_size=100000, output_dir='/content/drive/MyDrive/processed_log_segments'):
    """
    Generates token IDs, embeddings for semantic sentences using a pre-trained BERT model,
    calculates hybrid vectors, and saves processed log segments periodically.
    Resumes from the last saved segment if interrupted.

    Args:
        log_df (pd.DataFrame): DataFrame containing semantic sentences and uco_labels.
        model_name (str): The name or path of the BERT model to use.
        batch_size (int): The number of sentences to process in each batch.
        segment_size (int): Number of logs in each segment file.
        output_dir (str): Directory to save processed log segments.
                          Segments will be saved as output_dir/processed_logs_segment_SEGMENT_NUMBER.parquet
                          where SEGMENT_NUMBER is an incrementing number.

    Returns:
        pd.DataFrame: Original DataFrame with new columns for token IDs, Cyber BERT embeddings and hybrid vectors added
                      for the processed segments. Note: This function modifies the input DataFrame in place
                      for the processed segments.
    """
    if log_df.empty or 'semantic_sentence' not in log_df.columns or log_df['semantic_sentence'].isnull().all() or 'uco_labels' not in log_df.columns:
        print("Required columns (semantic_sentence or uco_labels) not available for processing.")
        # Initialize columns with None for consistency, though they won't be filled if no data
        log_df['token_ids'] = [None] * len(log_df)
        log_df['cyber_bert_embedding'] = [None] * len(log_df)
        log_df['hybrid_vector'] = [None] * len(log_df)
        return log_df

    # Ensure output directory exists
    os.makedirs(output_dir, exist_ok=True)

    print(f"\n--- Loading Cybersecurity-aware BERT Tokenizer and Model ('{model_name}') ---")
    try:
        cyber_tokenizer = BertTokenizer.from_pretrained(model_name)
        # Set trust_remote_code=True if using a custom model from Hugging Face Hub
        cyber_model = BertModel.from_pretrained(model_name).to(device)
        cyber_model.eval()
    except Exception as e:
        print(f"Error loading BERT model: {e}")
        # Initialize columns with None if model loading fails
        log_df['token_ids'] = [None] * len(log_df)
        log_df['cyber_bert_embedding'] = [None] * len(log_df)
        log_df['hybrid_vector'] = [None] * len(log_df)
        return log_df

    print(f"\n--- Generating Token IDs, Cyber BERT Embeddings and Hybrid Vectors in batches of {batch_size} and saving segments of {segment_size} logs ---")

    sentences = log_df['semantic_sentence'].tolist()
    uco_labels = log_df['uco_labels'].tolist()
    num_sentences = len(sentences)

    # Initialize MultiLabelBinarizer with all possible UCO labels from the dataframe
    mlb = MultiLabelBinarizer()
    # Fit on all uco_labels to ensure consistent encoding across segments
    all_possible_uco_labels = [item for sublist in uco_labels for item in sublist]
    mlb.fit([all_possible_uco_labels])

    # Initialize columns for token IDs, embeddings and hybrid vectors if they don't exist
    if 'token_ids' not in log_df.columns:
        log_df['token_ids'] = [None] * num_sentences
    if 'cyber_bert_embedding' not in log_df.columns:
         log_df['cyber_bert_embedding'] = [None] * num_sentences
    if 'hybrid_vector' not in log_df.columns:
        log_df['hybrid_vector'] = [None] * num_sentences


    # Determine the starting index for processing and the next segment number
    start_index = 0
    last_segment_number = 0
    # Find existing segment files to resume from
    segment_files = [f for f in os.listdir(output_dir) if re.match(r'processed_logs_segment_\d+\.parquet', f)]

    if segment_files:
        print(f"Found existing segment files in {output_dir}: {segment_files}")
        # Extract the segment number from the filenames and find the largest to determine where to resume
        segment_numbers = []
        for f in segment_files:
            match = re.search(r'_(\d+)\.parquet', f)
            if match:
                try:
                    segment_num = int(match.group(1))
                    segment_numbers.append(segment_num)
                except ValueError:
                    print(f"Skipping segment file with invalid segment number: {f}")

        if segment_numbers:
            last_segment_number = max(segment_numbers)
            # Calculate the start index based on the last segment number and size
            # Note: This assumes each saved segment except potentially the last is exactly segment_size.
            # A more robust approach would be to load the last segment to get its exact length,
            # but to save memory, we'll rely on the segment size assumption for the start index calculation.
            start_index = last_segment_number * segment_size
            print(f"Resuming processing from log index {start_index} (after processing segment {last_segment_number}).")
        else:
            print("No valid segment files found to resume from. Starting from the beginning.")

    if start_index >= num_sentences:
        print("All logs have already been processed based on existing segment files.")
        return log_df


    start_time = time.time() # Start time
    current_segment_number = last_segment_number + 1

    # No need to load existing segments into RAM.
    # The processing loop will start from `start_index` and append to the original DataFrame.


    # Use tqdm for a progress bar, starting from the resumed index
    with tqdm(total=num_sentences, initial=start_index, desc="Generating Token IDs, Embeddings and Hybrid Vectors") as pbar:
        for i in range(start_index, num_sentences, batch_size):
            batch_sentences = sentences[i : i + batch_size]
            batch_uco_labels = uco_labels[i : i + batch_size]

            # Generate token IDs for the batch
            batch_tokenized = cyber_tokenizer(
                batch_sentences,
                add_special_tokens=True,
                max_length=128, # Adjust max_length as needed based on your sentences and model
                padding='max_length',
                truncation=True,
                return_tensors='pt'
            )
            batch_token_ids = batch_tokenized['input_ids'].tolist()


            # Generate BERT embeddings for the batch
            with torch.no_grad():
                # Move inputs to the device
                batch_inputs = {key: val.to(device) for key, val in batch_tokenized.items()}
                outputs = cyber_model(**batch_inputs)
                # Extract the [CLS] token embedding for the batch
                batch_cls_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()


            # Calculate hybrid vectors for the batch
            uco_encoded_batch = mlb.transform(batch_uco_labels)
            batch_hybrid_vectors = [np.concatenate((bert_emb, uco_enc)) for bert_emb, uco_enc in zip(batch_cls_embeddings, uco_encoded_batch)]

            # Assign the generated token IDs, embeddings and hybrid vectors directly to the log_df slice
            end_index = i + len(batch_sentences)
            # Assign as a list of objects to avoid ValueError
            log_df.loc[i:end_index-1, 'token_ids'] = pd.Series(batch_token_ids, index=log_df.index[i:end_index])
            log_df.loc[i:end_index-1, 'cyber_bert_embedding'] = pd.Series(batch_cls_embeddings.tolist(), index=log_df.index[i:end_index])
            log_df.loc[i:end_index-1, 'hybrid_vector'] = pd.Series([vec for vec in batch_hybrid_vectors], index=log_df.index[i:end_index])


            # Update progress bar
            pbar.update(len(batch_sentences))

            # Save segment periodically
            # Check if the current index (end of the batch) is at or beyond the next segment boundary
            current_processed_count = i + len(batch_sentences)
            next_segment_boundary = current_segment_number * segment_size

            # Save if we've reached or passed a segment boundary or if it's the last batch
            if current_processed_count >= next_segment_boundary or current_processed_count == num_sentences:
                 segment_start = (current_segment_number - 1) * segment_size
                 segment_end = current_processed_count

                 # Create a DataFrame for the segment from the original log_df slice
                 segment_df = log_df.iloc[segment_start : segment_end].copy()

                 # Ensure required columns for saving are present - already done when adding
                 # cols_to_save = ['token_ids', 'cyber_bert_embedding', 'hybrid_vector']
                 # segment_df = segment_df[[col for col in cols_to_save if col in segment_df.columns]]

                 if not segment_df.empty:
                     segment_path = os.path.join(output_dir, f'processed_logs_segment_{current_segment_number}.parquet')
                     try:
                        # Keep only the desired columns if they exist
                        cols_to_save = [col for col in ['log', 'uco_labels', 'hybrid_vector'] if col in segment_df.columns]
                        segment_df = segment_df[cols_to_save]

                        # Save only selected columns
                        segment_df.to_parquet(segment_path, index=False)
                        tqdm.write(f"Segment {current_segment_number} (logs {segment_start}–{segment_end}) saved with columns: {cols_to_save} to {segment_path}")

                        current_segment_number += 1  # Increment segment number for the next segment

                     except Exception as e:
                         tqdm.write(f"Error saving segment {current_segment_number} for logs {segment_start} to {segment_end}: {e}")

            # Calculate and print speed periodically (e.g., every 10 batches)
            if (i // batch_size + 1) % 10 == 0 or current_processed_count == num_sentences:
                elapsed_time = time.time() - start_time
                processed_count_since_start = current_processed_count - start_index
                speed = processed_count_since_start / elapsed_time if elapsed_time > 0 else 0
                tqdm.write(f"Processed {current_processed_count}/{num_sentences} total logs in {elapsed_time:.2f} seconds ({speed:.2f} sentences/sec)")


    # The results are already assigned to the original log_df slices during the loop.
    # No need to assign from collection lists at the end.

    return log_df

# Generate embeddings and hybrid vectors for the semantic sentences
# Specify segment_size and the directory to save segments
# The function will automatically find and load the latest segment in the directory to resume from.
enriched_log_df = generate_cyber_bert_embeddings_and_hybrid_vectors(enriched_log_df, segment_size=50000, output_dir='/content/drive/MyDrive/processed_log_segments', batch_size=512)

# Display the first few rows with the new embedding and hybrid vector columns
print("\n--- Enriched Log Data with Token IDs, Cyber BERT Embeddings and Hybrid Vectors ---")
if not enriched_log_df.empty and 'token_ids' in enriched_log_df.columns and 'cyber_bert_embedding' in enriched_log_df.columns and 'hybrid_vector' in enriched_log_df.columns:
    # Select columns to display, ensuring they exist and are not None for the head rows
    display_cols = ['semantic_sentence', 'token_ids', 'cyber_bert_embedding', 'hybrid_vector']
    existing_display_cols = [col for col in display_cols if col in enriched_log_df.columns]
    # Filter out rows where these columns might be None if processing was interrupted
    display_df = enriched_log_df[existing_display_cols].dropna(subset=['token_ids', 'cyber_bert_embedding', 'hybrid_vector'])
    print(display_df.head())
    print(f"\nTotal number of logs with Token IDs, Cyber BERT embeddings and hybrid vectors: {len(display_df)}")
else:
    print("Token IDs, Cyber BERT embeddings or hybrid vectors were not generated or are empty.")



In [ ]:
# Cell 7: Save Processed Data

# Function to save the enriched DataFrame to a file
def save_enriched_data(log_df, file_path):
    """
    Saves only the 'hybrid_vector', 'log', and 'actions' columns from the enriched log DataFrame
    to a specified Parquet file.

    Args:
        log_df (pd.DataFrame): DataFrame containing enriched log data.
        file_path (str): The path where the DataFrame will be saved.
    """
    if log_df.empty:
        print("No data to save.")
        return

    print(f"\n--- Saving Selected Log Data to {file_path} ---")
    try:
        # Keep only these three columns if they exist
        cols_to_save = [col for col in ['hybrid_vector', 'log', 'actions'] if col in log_df.columns]

        if not cols_to_save:
            print("None of the required columns ('hybrid_vector', 'log', 'actions') were found in the DataFrame.")
            return

        # Create a smaller DataFrame containing only the selected columns
        df_to_save = log_df[cols_to_save]

        print(f"Saving only selected columns: {cols_to_save}")

        # Save as Parquet file
        df_to_save.to_parquet(file_path, index=False)

        print("Selected log data saved successfully.")
        print(f"File saved to: {file_path}")

    except Exception as e:
        print(f"Error saving enriched data: {e}")


# Example usage: Replace with your desired output file path
output_file_path = '/content/drive/MyDrive/enriched_logs_with_cyber_embeddings.parquet'
save_enriched_data(enriched_log_df, output_file_path)
